# Silver Layer — Payment Events

**Table**: `aiubereats.payments.silver_payment_events`  
**Source**: `aiubereats.payments.bronze_payment_events`  
**Purpose**: Cleansed, conformed, deduplicated, schema-enforced single source of truth.  
**Quality level**: Validated, typed, nulls handled, PII-clean.

## Transformations Applied

| Step | Transformation | Justification |
|------|---------------|---------------|
| 1 | Deduplicate on `event_id` | Payment events are idempotent by `event_id`. Duplicate rows from re-ingestion or source fan-out must not produce duplicate records in Silver. Strategy: keep the row with the latest `_ingested_at`. |
| 2 | Cast `event.timestamp` `BIGINT` → `TIMESTAMP` | Epoch milliseconds ÷ 1000 → epoch seconds → `timestamp_seconds()`. Done in Silver, not Bronze, because casting is a transformation. |
| 3 | Parse `dt_current_timestamp` `STRING` → `TIMESTAMP` | Source delivers `"2025-10-05 18:06:40.420"`. Cast with `to_timestamp()` using the exact format string to be deterministic and locale-safe. |
| 4 | Flatten `event` struct → `event_name`, `event_timestamp` | Struct access is verbose for analysts. Flattening to scalar columns enables direct filtering and partitioning without dot-notation. |
| 5 | Null handling: reject rows where `event_id` or `payment_id` is null | These are the business keys. A row without them cannot be meaningfully deduplicated or joined. Such rows go to `quarantine`. |
| 6 | Validate `event_name` enum | Only `created`, `authorized`, `captured` are valid. Other values indicate upstream corruption and go to `quarantine`. |
| 7 | MERGE INTO Silver | Upsert on `event_id`. Handles re-runs and late-arriving corrections without duplicates. |

## Deduplication Strategy

Business key: `event_id` (globally unique per event).  
Within a dedup window (`PARTITION BY event_id ORDER BY _ingested_at DESC`), `ROW_NUMBER() = 1` selects the most-recently-ingested copy.  
This handles source systems that re-emit the same event_id (idempotent re-delivery).  
After dedup, the resulting set is MERGEd into Silver so that corrections from a re-ingested batch update existing Silver rows rather than creating duplicates.

## Serverless / Spark Connect Notes

- `Window` functions (row_number, first, lag) are fully supported on Spark Connect.
- `DeltaTable.forName()` and `.merge()` work on Serverless via the `delta` package (bundled in Databricks Runtime).
- `timestamp_seconds()` is a Spark SQL function available as `F.timestamp_seconds()` — no RDD required.
- No `sparkContext`, no `mapPartitions`, no accumulators used anywhere in this notebook.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, TimestampType
)
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

In [ ]:
# ── Configuration — Databricks Widgets ───────────────────────────────────────────────
#
# Widget declarations are idempotent in Databricks — safe to re-run.

dbutils.widgets.text("bronze_table",     "aiubereats.payments.bronze_payment_events")
dbutils.widgets.text("silver_table",     "aiubereats.payments.silver_payment_events")
dbutils.widgets.text("quarantine_table", "aiubereats.payments.quarantine_payment_events")

BRONZE_TABLE     = dbutils.widgets.get("bronze_table")
SILVER_TABLE     = dbutils.widgets.get("silver_table")
QUARANTINE_TABLE = dbutils.widgets.get("quarantine_table")

BUSINESS_KEY      = "event_id"
VALID_EVENT_NAMES = {"created", "authorized", "captured"}
DT_FORMAT         = "yyyy-MM-dd HH:mm:ss.SSS"   # matches "2025-10-05 18:06:40.420"

In [ ]:
# ── Create Silver table (DDL) ─────────────────────────────────────────────────
#
# Liquid Clustering replaces traditional PARTITIONED BY on Databricks Serverless.
# PARTITIONED BY + CLUSTER BY in the same DDL raises SPECIFY_CLUSTER_BY_WITH_PARTITIONED_BY_IS_NOT_ALLOWED.
#
# CLUSTER BY (event_date, payment_id, event_name):
#   - event_date: enables efficient date-range scans by Gold and ad-hoc queries.
#   - payment_id: supports the Gold layer JOIN/pivot on payment lifecycle.
#   - event_name: optimises the Gold conditional-aggregation filter.
#
# enableChangeDataFeed: enables CDF so Gold can read only changed Silver rows
# in future incremental runs, avoiding full Silver scans.

spark.sql("""
    CREATE TABLE IF NOT EXISTS aiubereats.payments.silver_payment_events (
        event_id               STRING     NOT NULL,
        payment_id             STRING     NOT NULL,
        event_name             STRING     NOT NULL,
        event_timestamp        TIMESTAMP  NOT NULL,
        dt_current_timestamp   TIMESTAMP,
        event_date             DATE       NOT NULL,
        _ingested_at           TIMESTAMP,
        _source_file           STRING,
        _source_system         STRING,
        _silver_processed_at   TIMESTAMP
    )
    USING DELTA
    CLUSTER BY (event_date, payment_id, event_name)
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite'       = 'true',
        'delta.autoOptimize.autoCompact'         = 'true',
        'delta.enableDeletionVectors'            = 'true',
        'delta.enableChangeDataFeed'             = 'true',
        'delta.logRetentionDuration'             = 'interval 60 days',
        'delta.deletedFileRetentionDuration'     = 'interval 7 days'
    )
    COMMENT 'Silver layer: cleansed, deduplicated, type-cast payment event records.'
""")

In [ ]:
# ── Create Quarantine table ───────────────────────────────────────────────────
#
# Records that fail Silver quality gates land here for investigation.
# Quarantine is append-only; it captures the raw Bronze columns plus
# a _quarantine_reason column describing which check failed.

spark.sql("""
    CREATE TABLE IF NOT EXISTS aiubereats.payments.quarantine_payment_events (
        event_id              STRING,
        payment_id            STRING,
        event                 STRUCT<event_name: STRING, timestamp: BIGINT>,
        dt_current_timestamp  STRING,
        _ingested_at          TIMESTAMP,
        _source_file          STRING,
        _source_system        STRING,
        _quarantine_reason    STRING,
        _quarantined_at       TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite'   = 'true',
        'delta.autoOptimize.autoCompact'     = 'true',
        'delta.logRetentionDuration'         = 'interval 90 days'
    )
    COMMENT 'Quarantine: Bronze records that failed Silver quality gates.'
""")

In [ ]:
# ── Step 1: Read from Bronze ──────────────────────────────────────────────────
#
# Read all Bronze records. In a production incremental run this would be
# filtered by _ingested_at > max(_silver_processed_at) from Silver, but
# for initial load we read everything.

bronze_df = spark.table(BRONZE_TABLE)
print(f"Bronze records to process: {bronze_df.count():,}")

In [ ]:
# ── Step 2: Quality gate — classify valid vs quarantine rows ──────────────────
#
# Rule 1 — event_id not null  (business key; without it dedup is impossible)
# Rule 2 — payment_id not null (FK to payment entity; required for Gold join)
# Rule 3 — event struct not null
# Rule 4 — event.event_name in allowed set
# Rule 5 — event.timestamp not null (needed for lifecycle timing in Gold)
#
# _quarantine_reason captures the FIRST failing check per row (priority order).
# Rows passing all checks proceed to Silver.

valid_names_list = list(VALID_EVENT_NAMES)  # for isin()

checked_df = (
    bronze_df
    .withColumn(
        "_quarantine_reason",
        F.when(F.col("event_id").isNull(),          F.lit("event_id is null"))
         .when(F.col("payment_id").isNull(),         F.lit("payment_id is null"))
         .when(F.col("event").isNull(),              F.lit("event struct is null"))
         .when(
             ~F.col("event.event_name").isin(valid_names_list),
             F.concat(
                 F.lit("invalid event_name: "),
                 F.coalesce(F.col("event.event_name"), F.lit("<null>"))
             )
         )
         .when(F.col("event.timestamp").isNull(),    F.lit("event.timestamp is null"))
         .otherwise(F.lit(None).cast(StringType()))
    )
)

valid_df      = checked_df.filter(F.col("_quarantine_reason").isNull()).drop("_quarantine_reason")
quarantine_df = checked_df.filter(F.col("_quarantine_reason").isNotNull())

valid_count      = valid_df.count()
quarantine_count = quarantine_df.count()

print(f"Valid records:      {valid_count:,}")
print(f"Quarantine records: {quarantine_count:,}")

if quarantine_count > 0:
    print("\nQuarantine breakdown:")
    quarantine_df.groupBy("_quarantine_reason").count().display()

In [ ]:
# ── Step 3: Write quarantined records ─────────────────────────────────────────

if quarantine_count > 0:
    (
        quarantine_df
        .withColumn("_quarantined_at", F.current_timestamp())
        # Keep only the Bronze columns that exist in quarantine schema
        .select(
            "event_id", "payment_id", "event", "dt_current_timestamp",
            "_ingested_at", "_source_file", "_source_system",
            "_quarantine_reason", "_quarantined_at"
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(QUARANTINE_TABLE)
    )
    print(f"{quarantine_count:,} records written to {QUARANTINE_TABLE}")

In [ ]:
# ── Step 4: Transform valid records ──────────────────────────────────────────
#
# event.timestamp (BIGINT, epoch milliseconds) → TIMESTAMP:
#   timestamp_seconds(event.timestamp / 1000)
#   Division by 1000 converts ms → seconds. timestamp_seconds() then
#   builds a TIMESTAMP from epoch seconds (accepts fractional seconds).
#   This is fully Spark Connect compatible — no UDF, no RDD.
#
# dt_current_timestamp (STRING) → TIMESTAMP:
#   to_timestamp() with an explicit format string.
#   Explicit format prevents locale-dependent parsing failures.
#
# event_date: DATE extracted from event_timestamp, used as partition column.

transformed_df = (
    valid_df
    # Flatten the event struct
    .withColumn("event_name",
        F.col("event.event_name")
    )
    .withColumn("event_timestamp",
        F.timestamp_seconds(
            (F.col("event.timestamp") / 1000).cast("double")
        )
    )
    # Parse source string timestamp
    .withColumn("dt_current_timestamp",
        F.to_timestamp(F.col("dt_current_timestamp"), DT_FORMAT)
    )
    # Derive event_date partition column
    .withColumn("event_date",
        F.to_date(F.col("event_timestamp"))
    )
    # Add Silver processing metadata
    .withColumn("_silver_processed_at", F.current_timestamp())
    # Drop the raw struct — Silver is flat
    .drop("event")
    # Select final Silver column order
    .select(
        "event_id",
        "payment_id",
        "event_name",
        "event_timestamp",
        "dt_current_timestamp",
        "event_date",
        "_ingested_at",
        "_source_file",
        "_source_system",
        "_silver_processed_at",
    )
)

In [ ]:
# ── Step 5: Deduplicate on event_id ──────────────────────────────────────────
#
# Within the current batch, keep the row with the latest _ingested_at per event_id.
# This handles the case where the same JSON file was added to the Volume twice,
# or the source system re-emitted an event with corrected fields.
#
# WHY ROW_NUMBER over dropDuplicates:
#   dropDuplicates("event_id") picks an arbitrary row when there are ties.
#   ROW_NUMBER with ORDER BY _ingested_at DESC is deterministic: it always
#   selects the most recent ingestion of a given event_id.
#
# Serverless note: Window functions run on the Spark cluster via Spark Connect;
# they do not require any driver-side RDD operations.

dedup_window = (
    Window
    .partitionBy(BUSINESS_KEY)
    .orderBy(F.col("_ingested_at").desc())
)

deduped_df = (
    transformed_df
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

deduped_count = deduped_df.count()
print(f"Records after deduplication: {deduped_count:,} (removed {valid_count - deduped_count:,} duplicates)")

In [ ]:
# ── Step 6: MERGE into Silver ─────────────────────────────────────────────────
#
# MERGE ON event_id:
#   WHEN MATCHED AND source._ingested_at > target._ingested_at → UPDATE
#     Only update if the incoming record is newer. This prevents a late-arriving
#     older Bronze batch from overwriting a corrected Silver record.
#   WHEN NOT MATCHED → INSERT
#     New event_ids (first time seen) are inserted.
#
# This pattern is idempotent: re-running the notebook with the same Bronze data
# produces no net change to Silver.

if spark.catalog.tableExists(SILVER_TABLE):
    silver_delta = DeltaTable.forName(spark, SILVER_TABLE)

    (
        silver_delta.alias("t")
        .merge(
            deduped_df.alias("s"),
            "t.event_id = s.event_id"
        )
        .whenMatchedUpdate(
            condition="s._ingested_at > t._ingested_at",
            set={
                "event_name":            "s.event_name",
                "event_timestamp":       "s.event_timestamp",
                "dt_current_timestamp":  "s.dt_current_timestamp",
                "event_date":            "s.event_date",
                "_ingested_at":          "s._ingested_at",
                "_source_file":          "s._source_file",
                "_source_system":        "s._source_system",
                "_silver_processed_at":  "s._silver_processed_at",
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE complete into {SILVER_TABLE}")
else:
    # First run — table was created by DDL but has no data yet
    (
        deduped_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Initial load complete into {SILVER_TABLE}")

In [ ]:
# ── Verify ────────────────────────────────────────────────────────────────────
silver = spark.table(SILVER_TABLE)
print(f"Silver row count: {silver.count():,}")
silver.display()

In [ ]:
# ── Post-Silver OPTIMIZE ──────────────────────────────────────────────────────
#
# OPTIMIZE compacts small Delta files produced by the MERGE and applies
# liquid clustering on (payment_id, event_name).
# On Databricks Serverless, autoOptimize runs in the background, but an
# explicit OPTIMIZE after the initial load is still good practice to ensure
# Gold reads have optimal file layout.

spark.sql(f"OPTIMIZE {SILVER_TABLE}")
print("OPTIMIZE complete.")